# 🚀 OmniCut Pro Studio + AGY Cloud Agent — Master Cloud Runner
### ระบบผลิตและตัดต่อคลิปวิดีโอ AI อัตโนมัติ พร้อม AGY Interactive Web Terminal (FastAPI + React SPA + HyperFrames + Gemini Flash TTS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazirulhak000/Omnicut/blob/main/Omnicut_Cloud_Runner.ipynb)

> ⚡ **วิธีเปิดใช้งาน:** กดปุ่ม **▶ (Play)** ที่เซลล์ด้านล่างเพียงปุ่มเดียว ระบบจะติดตั้ง ซิงก์โค้ด ตั้งค่าคีย์ เปิดหน้าเว็บสตูดิโอ และเปิด **AGY Cloud Terminal (`/agy`)** ให้ทันทีภายในไม่กี่วินาที พร้อมระบบ **Keepalive ป้องกันการหลับ** ให้อัตโนมัติ!


In [ ]:
# @title 🚀 คลิกปุ่มนี้ปุ่มเดียว: เปิดระบบ OmniCut Pro Studio + AGY Cloud Terminal + Keepalive กันหลับ { vertical-output: true }
# @markdown Cloudflare Tunnel Token (ตั้งค่าให้พร้อมใช้งานสำหรับ studio.nazirulhak.com แล้ว)
CLOUDFLARE_TUNNEL_TOKEN = "eyJhIjoiNjQ3NDU3MTM3ZjBlZTdhMzc3MmFlNGEzOWE3OWJkYjciLCJ0IjoiYmFmZmM5MzMtZWRkMy00MDE2LWI4NGMtNzcwMGRhZjZjMTQ0IiwicyI6IlpESmpaR1UyTURJdFptUmpNQzAwT0RGbExUZ3dZVEF0TjJVeU1qSXdOakl3WkRBNSJ9"  # @param {type:"string"}

import os
import sys
import time
import subprocess
import datetime
import re
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/Omnicut') if IS_COLAB else Path.cwd()
print('🚀 [1/5] เริ่มต้นระบบ OmniCut Pro Studio + AGY Cloud Agent...')

# ----------------------------------------------------------------
# 1. ซิงก์โค้ดล่าสุดจาก GitHub
# ----------------------------------------------------------------
if IS_COLAB:
    if not REPO_DIR.exists():
        print('  📂 กำลังโคลนโค้ด OmniCut จาก GitHub...')
        os.system(f'git clone -q https://github.com/nazirulhak000/Omnicut.git {REPO_DIR}')
    else:
        print('  🔄 อัปเดตโค้ดล่าสุด (git pull origin main)...')
        os.system(f'cd "{REPO_DIR}" && git fetch -q origin && git reset -q --hard origin/main')
    (REPO_DIR / 'scripts' / '__init__.py').touch()
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')
    sync_tool = REPO_DIR / 'tools' / 'sync_scripts_to_products.py'
    if sync_tool.exists():
        print('  ✨ ซิงก์ฐานข้อมูลและสคริปต์ 180 บทเข้าสู่ระบบ...')
        os.system(f'cd "{REPO_DIR}" && {sys.executable} "{sync_tool}"')

# ----------------------------------------------------------------
# 2. ตรวจสอบ System Packages, Python Dependencies & AGY CLI
# ----------------------------------------------------------------
print('\n📦 [2/5] ตรวจสอบความพร้อมของแพ็กเกจระบบและ AGY Cloud Agent...')
if IS_COLAB:
    req_file = REPO_DIR / 'requirements.txt'
    if req_file.exists():
        os.system(f'{sys.executable} -m pip install -q -r "{req_file}" aiofiles')
    else:
        os.system(f'{sys.executable} -m pip install -q fastapi "uvicorn[standard]" python-multipart aiofiles pythainlp soundfile faster-whisper')
    
    # ติดตั้ง google-antigravity SDK (ถ้าแพลตฟอร์มรองรับ)
    os.system(f'{sys.executable} -m pip install -q google-antigravity > /dev/null 2>&1 || true')
    
    # ติดตั้ง cloudflared ถ้ายังไม่มี
    cf_bin = Path('/usr/local/bin/cloudflared')
    if not cf_bin.exists():
        print('  ⚡ ติดตั้ง Cloudflare Tunnel binary...')
        os.system('curl -sSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
    
    # ติดตั้ง ttyd Interactive Web Terminal binary
    ttyd_bin = Path('/usr/local/bin/ttyd')
    if not ttyd_bin.exists():
        print('  🖥️ ติดตั้ง ttyd Interactive Web Terminal binary...')
        os.system('curl -sSL https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64 -o /usr/local/bin/ttyd && chmod +x /usr/local/bin/ttyd')
    
    # กู้คืนเซสชันล็อกอิน Antigravity (~/.gemini) จาก Google Drive ถาวร
    drive_gemini = Path('/content/drive/MyDrive/.gemini')
    if drive_gemini.exists():
        print('  ✨ กู้คืนเซสชันล็อกอิน Antigravity จาก Google Drive สำเร็จ!')
        os.system('mkdir -p /root/.gemini && cp -rn /content/drive/MyDrive/.gemini/* /root/.gemini/ > /dev/null 2>&1 || true')
    
    # ติดตั้ง Official Google Antigravity CLI (agy) Native Binary
    official_agy = Path('/root/.local/bin/agy')
    if not official_agy.exists():
        print('  🤖 ติดตั้ง Official Google Antigravity CLI (agy) Native Binary...')
        os.system('curl -fsSL https://antigravity.google/cli/install.sh | bash > /tmp/agy_install.log 2>&1 || true')
    
    if official_agy.exists():
        os.system('ln -sf /root/.local/bin/agy /usr/local/bin/agy && chmod +x /usr/local/bin/agy')
        print('  ✅ ติดตั้ง Official Google Antigravity CLI สำเร็จ (พร้อมล็อกอินผ่าน Google Account)!')
    else:
        agy_bin = Path('/usr/local/bin/agy')
        agy_bin.write_text(f'#!/bin/bash\ncd "{REPO_DIR}" && {sys.executable} "{REPO_DIR}/scripts/colab_agy_cli.py" "$@"\n', encoding='utf-8')
        os.system('chmod +x /usr/local/bin/agy')
        print('  ⚡ ใช้ AGY Cloud Agent Python CLI สำรองเรียบร้อย!')
print('  ✅ Dependencies & AGY CLI พร้อมทำงาน 100%')

# ----------------------------------------------------------------
# 3. โหลดและตั้งค่า Gemini API Keys (Colab Secrets / Drive / .env)
# ----------------------------------------------------------------
print('\n🔑 [3/5] ตั้งค่า Gemini API Keys (Colab Secrets / Drive / .env)...')
active_keys = []
try:
    from google.colab import userdata
    for secret_name in ['GEMINI_API_KEY', 'GEMINI_API_KEY_2', 'GEMINI_API_KEY_3', 'GOOGLE_API_KEY']:
        try:
            uk = userdata.get(secret_name)
            if uk and uk.strip() and uk.strip() not in active_keys:
                active_keys.append(uk.strip())
        except Exception:
            pass
    if active_keys:
        print(f'  ✨ ดึงคีย์จาก Google Colab Secrets (รูปแม่กุญแจ 🔑) สำเร็จ ({len(active_keys)} คีย์)')
except Exception:
    pass

# ตรวจสอบไฟล์ .env บน Google Drive หรือในโฟลเดอร์โปรเจกต์
drive_env = Path('/content/drive/MyDrive/Omnicut/.env')
if drive_env.exists() and not active_keys:
    for line in drive_env.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if line.startswith('GEMINI_API_KEY') or line.startswith('GOOGLE_API_KEY'):
            if '=' in line:
                v = line.split('=', 1)[1].strip().strip('"').strip("'")
                if v and v not in active_keys:
                    active_keys.append(v)
    if active_keys:
        print(f'  ✨ โหลดคีย์จาก Google Drive ({drive_env}) สำเร็จ ({len(active_keys)} คีย์)')

colab_env = REPO_DIR / '.env'
if colab_env.exists() and not active_keys:
    for line in colab_env.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if line.startswith('GEMINI_API_KEY') or line.startswith('GOOGLE_API_KEY'):
            if '=' in line:
                v = line.split('=', 1)[1].strip().strip('"').strip("'")
                if v and v not in active_keys:
                    active_keys.append(v)

if not active_keys:
    print('  ⚠️ ไม่พบคีย์ใน Colab Secrets หรือ Google Drive!')
    print('  👉 กรุณาเพิ่มคีย์ที่เมนู Secrets (รูปแม่กุญแจ 🔑 ด้านซ้าย) ชื่อ GEMINI_API_KEY')
    try:
        import getpass
        prompted_key = getpass.getpass('🔑 หรือวาง GEMINI_API_KEY ที่นี่ (ซ่อนตัวอักษร): ').strip()
        if prompted_key:
            active_keys.append(prompted_key)
    except Exception:
        pass

# ซิงก์คีย์ลง Environment และไฟล์ .env สำหรับ FastAPI / AGY Cloud Agent
env_lines = []
for i, k in enumerate(active_keys):
    var_name = 'GEMINI_API_KEY' if i == 0 else f'GEMINI_API_KEY_{i+1}'
    os.environ[var_name] = k
    env_lines.append(f'{var_name}={k}')

if env_lines:
    colab_env.write_text('\n'.join(env_lines) + '\n', encoding='utf-8')
    print(f'  ✅ โหลดคีย์พร้อมใช้งาน {len(active_keys)} คีย์ (ซิงก์สู่ .env เรียบร้อย)!')
else:
    print('  ⚠️ ยังไม่มีคีย์ API — ระบบจะรอให้คุณใส่ผ่านเมนู Secrets หรือ Web Terminal')

# ----------------------------------------------------------------
# 4. เปิดเซิร์ฟเวอร์ FastAPI Backend (พอร์ต 8000) & ttyd Terminal (พอร์ต 7681)
# ----------------------------------------------------------------
print('\n🎬 [4/5] เปิดเซิร์ฟเวอร์ OmniCut Pro Studio Backend & AGY Terminal...')
os.system('fuser -k 8000/tcp > /dev/null 2>&1')
os.system('fuser -k 7681/tcp > /dev/null 2>&1')
os.system('pkill -f cloudflared > /dev/null 2>&1')

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR)
env['OMNICUT_ALLOW_REMOTE'] = '1'
fastapi_log = open('/tmp/fastapi.log', 'w')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'scripts.api_server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=str(REPO_DIR),
    env=env,
    stdout=fastapi_log,
    stderr=subprocess.STDOUT,
)
if IS_COLAB and Path('/usr/local/bin/ttyd').exists():
    subprocess.Popen(
        ['/usr/local/bin/ttyd', '-W', '-p', '7681', 'bash', '-c', 'cd ' + str(REPO_DIR) + ' && agy || bash'],
        cwd=str(REPO_DIR),
        env=env,
        stdout=open('/tmp/ttyd.log', 'w'),
        stderr=subprocess.STDOUT,
    )
time.sleep(3)

# ทดสอบว่า FastAPI ตอบสนอง 200 OK หรือไม่
server_ok = False
for _ in range(20):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/ai/status', timeout=1) as r:
            if r.status == 200:
                server_ok = True
                break
    except Exception:
        time.sleep(0.5)

if server_ok:
    print('  ✅ OmniCut FastAPI Server & AGY Web Terminal (/agy) ทำงานบนพอร์ต 8000 สมบูรณ์')
else:
    print('  ⚠️ ตรวจสอบบันทึก /tmp/fastapi.log:')
    if os.path.exists('/tmp/fastapi.log'):
        print(open('/tmp/fastapi.log').read()[-500:])

# ----------------------------------------------------------------
# 5. เชื่อมต่ออุโมงค์สัญญาณ Cloudflare Tunnel & แสดงลิงก์
# ----------------------------------------------------------------
print('\n🌐 [5/5] กำลังเชื่อมต่อ Cloudflare Tunnel...')
tunnel_log = '/tmp/cloudflared.log'
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)

if CLOUDFLARE_TUNNEL_TOKEN.strip():
    print('  ⚡ ใช้ Named Cloudflare Tunnel (ถาวร)...')
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TUNNEL_TOKEN.strip()],
        stdout=open(tunnel_log, 'w'),
        stderr=subprocess.STDOUT,
    )
    public_url = 'https://studio.nazirulhak.com'
else:
    print('  ⚡ ใช้ Instant Quick Tunnel (สร้างลิงก์สดทันที)...')
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=open(tunnel_log, 'w'),
        stderr=subprocess.STDOUT,
    )
    public_url = None
    for _ in range(35):
        time.sleep(1)
        if os.path.exists(tunnel_log):
            content = open(tunnel_log).read()
            match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    agy_url = f'{public_url.rstrip("/")}/agy'
    from IPython.display import display, HTML
    print('\n' + '=' * 70)
    print('🎉 OmniCut Pro Studio + AGY Cloud Terminal พร้อมใช้งานแล้ว 100%!')
    print(f'👉 ลิงก์สตูดิโอหลัก   : {public_url}')
    print(f'🤖 ลิงก์ AGY Terminal : {agy_url}')
    print('=' * 70 + '\n')
    display(HTML(f'''
    <div style="padding: 24px; background: linear-gradient(135deg, #090d16 0%, #171d31 100%); border-radius: 14px; color: white; font-family: sans-serif; text-align: center; border: 1.5px solid #3b82f6; box-shadow: 0 10px 30px rgba(0,0,0,0.6);">
        <h2 style="margin: 0 0 10px 0; color: #60a5fa; font-size: 24px;">🚀 OmniCut Pro Studio + AGY Cloud Agent (Online 100%)</h2>
        <p style="margin: 0 0 20px 0; color: #94a3b8; font-size: 14px;">สตูดิโอตัดต่อวิดีโอ TikTok 9:16 และคอนโซลสั่งงาน AI อัตโนมัติ (AGY) บน Cloud เต็มรูปแบบ</p>
        <div style="display: flex; gap: 14px; justify-content: center; flex-wrap: wrap;">
            <a href="{public_url}" target="_blank" style="display: inline-block; padding: 14px 30px; background: #2563eb; color: white; text-decoration: none; font-weight: bold; font-size: 15px; border-radius: 8px; box-shadow: 0 4px 14px rgba(37, 99, 235, 0.4);">
                ✨ เปิด OmniCut Pro Studio
            </a>
            <a href="{agy_url}" target="_blank" style="display: inline-block; padding: 14px 30px; background: #0d9488; color: white; text-decoration: none; font-weight: bold; font-size: 15px; border-radius: 8px; box-shadow: 0 4px 14px rgba(13, 148, 136, 0.4);">
                🤖 เปิด AGY Cloud Terminal (/agy)
            </a>
        </div>
        <div style="margin-top: 16px; font-size: 13px; color: #64748b;">
            Studio: <a href="{public_url}" target="_blank" style="color: #38bdf8;">{public_url}</a> &nbsp;|&nbsp; 
            AGY Terminal: <a href="{agy_url}" target="_blank" style="color: #2dd4bf;">{agy_url}</a>
        </div>
    </div>
    '''))
else:
    print('⚠️ ไม่สามารถรับ Tunnel URL ได้ กรุณาดูบันทึกที่ /tmp/cloudflared.log')

# ----------------------------------------------------------------
# 6. Watchdog & Heartbeat Loop (Keepalive ป้องกันการตัดการเชื่อมต่อ)
# ----------------------------------------------------------------
print('\n🛡️ [Keepalive Active] เปิดระบบ Watchdog เฝ้าระวังไม่ให้เซสชันหลับ...')
print('   (คุณสามารถปิดหน้าเว็บ Colab หรือปิดคอมได้เลย ระบบจะรันดูแลตัวเองใน Background ต่อเนื่อง)\n')
heartbeat_ticks = 0
while True:
    time.sleep(30)
    heartbeat_ticks += 1
    
    # เฝ้าระวัง Backend: ถ้า FastAPI ดับ ให้รีสตาร์ตตัวเองอัตโนมัติ (Self-Healing)
    backend_alive = False
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/ai/status', timeout=2) as r:
            if r.status == 200:
                backend_alive = True
    except Exception:
        backend_alive = False
        
    if not backend_alive:
        now = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'⚠️ [{now}] ตรวจพบ Backend หยุดทำงาน กำลังรีสตาร์ตใหม่อัตโนมัติ...')
        fastapi_log = open('/tmp/fastapi.log', 'a')
        backend_proc = subprocess.Popen(
            [sys.executable, '-m', 'uvicorn', 'scripts.api_server:app', '--host', '0.0.0.0', '--port', '8000'],
            cwd=str(REPO_DIR),
            env=env,
            stdout=fastapi_log,
            stderr=subprocess.STDOUT,
        )
    
    # ซิงก์สำรองเซสชัน ~/.gemini สู่ Google Drive ทุก 5 นาที เมื่อล็อกอินแล้ว
    if heartbeat_ticks % 10 == 0 and Path('/root/.gemini').exists() and Path('/content/drive/MyDrive').exists():
        os.system('mkdir -p /content/drive/MyDrive/.gemini && cp -ru /root/.gemini/* /content/drive/MyDrive/.gemini/ > /dev/null 2>&1 || true')
    
    # ส่งสัญญาณ Heartbeat ทุก 5 นาที เพื่อให้ Google Colab รู้ว่ามี I/O Active อยู่ตลอดเวลา
    if heartbeat_ticks % 10 == 0:
        now = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'💓 [{now}] Heartbeat Active | OmniCut Studio + AGY Terminal พร้อมใช้งาน 100% | URL: {public_url}')


In [ ]:
# @title 🔑 ล็อกอินบัญชี Antigravity สำหรับ AGY CLI (ทำครั้งเดียว จำถาวรลง Google Drive) { vertical-output: true }
# @markdown รันเซลล์นี้หากต้องการล็อกอินบัญชี Google เข้ากับ AGY CLI โดยตรง (จะได้ URL สำหรับคลิกล็อกอินบนเบราว์เซอร์)
import os, sys
from pathlib import Path

# เรียกใช้คำสั่ง agy เพื่อแสดง URL ล็อกอิน
print('🤖 กำลังเปิด AGY CLI เพื่อเข้าสู่ระบบ Google Account...')
print('👉 หากระบบแสดงลิงก์ ให้คลิกลิงก์เพื่อล็อกอิน แล้วคัดลอกรหัสยืนยันมาวางในช่องด้านล่าง:\n')
os.system('cd /content/Omnicut && agy || true')

# สำรองไฟล์ล็อกอิน ~/.gemini สู่ Google Drive ถาวร
if Path('/root/.gemini').exists() and Path('/content/drive/MyDrive').exists():
    os.system('mkdir -p /content/drive/MyDrive/.gemini && cp -ru /root/.gemini/* /content/drive/MyDrive/.gemini/ 2>/dev/null || true')
    print('\n🎉 บันทึกเซสชันล็อกอิน AGY ลง Google Drive เรียบร้อย! ครั้งต่อไปจะล็อกอินให้อัตโนมัติ')
